In [1]:
# Imports and Configurations
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import time

# Clustering and distance metrics
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score

# Parallelization for OCP
import multiprocessing as mp

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

np.random.seed(42)

# Configuration
BENCHMARK        = '^GSPC'
RISK_FREE_RATE   = 0.045
START_DATE       = '2014-01-01'
END_DATE         = '2024-01-01'
ROLLING_WINDOW   = 52
MIN_HISTORY      = 400
K_VALUES         = [2,4,7]
N_CORES          = 14

# Three-period out-of-sample validation framework
TRAIN_START = '2014-01-01'
TRAIN_END   = '2017-12-31'
VAL_START   = '2018-01-01'
VAL_END     = '2020-12-31'
TEST_START  = '2021-01-01'
TEST_END    = '2023-12-31'

# Strategy Parameters
ZSCORE_WINDOW   = 52
VOL_WINDOW      = 26
VOL_THRESHOLD   = 1.5
HIGH_VOL_ENTRY  = 2.5
LOW_VOL_ENTRY   = 1.5
STANDARD_ENTRY  = 2.0
EXIT_THRESHOLD  = 0.5
STOP_LOSS       = 3.5

print('Notebook 6: DTW vs. OCP Clustering - S&P 500 Universe')
print(f'Period: {START_DATE} to {END_DATE}')
print(f'Risk-free rate: {RISK_FREE_RATE} (applied to both DTW and OCP Sharpe inputs)')
print(f'Parallelization: {N_CORES} cores for OCP')
print(f'\nThree-period framework (for later validation stage):')
print(f'    Train: {TRAIN_START} to {TRAIN_END}')
print(f'    Val:   {VAL_START} to {VAL_END}')
print(f'    Test:  {TEST_START} to {TEST_END}')
print(f'\nStrategy parameters (for later backtest stage):')
print(f'    Entry: ±{HIGH_VOL_ENTRY}σ (high vol) / ±{LOW_VOL_ENTRY}σ (low vol)')
print(f'    Exit:  ±{EXIT_THRESHOLD}σ')
print(f'    Stop:  ±{STOP_LOSS}σ')

Notebook 6: DTW vs. OCP Clustering - S&P 500 Universe
Period: 2014-01-01 to 2024-01-01
Risk-free rate: 0.045 (applied to both DTW and OCP Sharpe inputs)
Parallelization: 14 cores for OCP

Three-period framework (for later validation stage):
    Train: 2014-01-01 to 2017-12-31
    Val:   2018-01-01 to 2020-12-31
    Test:  2021-01-01 to 2023-12-31

Strategy parameters (for later backtest stage):
    Entry: ±2.5σ (high vol) / ±1.5σ (low vol)
    Exit:  ±0.5σ
    Stop:  ±3.5σ


In [ ]:
# Retrieving S&P500 constituents as of January 1, 2014

# This source avoids survivorship bias, since it will proviude